# 🧠 High on Therapy: 100% Bootstrapped Native Counseling AI
### Zero External Weights • Zero Qwen • From-Scratch Pre-training on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Suraj-Mavuleti/highontherapy-ai-training/blob/main/HighOnTherapy_Bootstrapped_From_Scratch.ipynb)

This notebook initializes a **100% proprietary Decoder-Only Transformer architecture** with completely randomized weights, trains a custom Byte-Level BPE tokenizer, and pre-trains the model exclusively on clinical psychotherapy and emotional counseling datasets.

### 📈 Scalable Architecture Tiers
- **Nano (~135M params)**: 12 layers, hidden 768. Perfect for free Google Colab T4 GPU (4–8 hours).
- **Base (~360M params)**: 16 layers, hidden 1024. Ideal for 12–24 hour runs on Colab Pro / A100 / L4.
- **Prime (~1.1B params)**: 22 layers, hidden 2048. Master clinical depth across multi-day runs.
- **Ultra (~3.2B params)**: 28 layers, hidden 3072. Advanced clinical specialization.

### 🛡️ Multi-Day Resilience: Google Drive Auto-Resume
All checkpoints are saved directly to your **Google Drive**. If Google Colab resets or times out, re-running the training cell **automatically detects the latest checkpoint and continues from the exact step and optimizer state**.



## 1. Environment Setup & Google Drive Mounting


In [ ]:
# 1. Mount Google Drive for persistent multi-day checkpointing
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/HighOnTherapy_Training'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f'Persistent checkpoint directory ready: {DRIVE_CHECKPOINT_DIR}')

# 2. Install dependencies
!pip install --quiet transformers tokenizers datasets accelerate huggingface_hub jinja2
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'GPU Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')



## 2. Hugging Face Authentication


In [ ]:
import os
import getpass
from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    print("Paste your Hugging Face write token below (it will be hidden as you type):")
    HF_TOKEN = getpass.getpass("HF Write Token: ").strip()

HF_REPO = "DeV-ZEr0/highontherapy-native"

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print(f"Authenticated with Hugging Face Hub. Target Repo: {HF_REPO}")
else:
    print("Running in local mode without automatic Hugging Face upload.")



## 3. Clinical Counseling Dataset Compilation


In [ ]:
import json
import urllib.request

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
CORPUS_FILE = os.path.join(DATA_DIR, "highontherapy_corpus.txt")
CHATML_FILE = os.path.join(DATA_DIR, "highontherapy_chatml.jsonl")

# Clinical frameworks and somatic protocols for High on Therapy
CLINICAL_PROTOCOLS = [
    {
        "user": "I feel completely frozen. I am staring at my laptop screen, unable to move, unable to reply to emails, and the more time passes the worse the panic gets.",
        "assistant": "What you are experiencing in this moment is a physiological dorsal vagal shutdown, commonly known as functional freeze. When your nervous system determines that the volume of demand has exceeded your biological capacity, it activates a primordial circuit breaker. The paralysis you feel is not a failure of character, laziness, or apathy; it is your autonomic nervous system attempting to protect you from perceived threat.\n\nTrying to force yourself into high productivity while in a freeze response only triggers more adrenaline and deepens the paralysis. Instead, we begin by communicating biological safety to your body. Allow your eyes to soften their focus. Notice the sensation of your feet firmly resting against the floor, and take a gentle, elongated exhale through your mouth, letting your shoulders drop.\n\nWithout expecting yourself to complete any tasks right now, what is the smallest physical sensation you can comfortably tune into in your body?"
    },
    {
        "user": "My heart is pounding in my chest, I can't catch my breath, and I feel like something terrible is about to happen right now.",
        "assistant": "I am right here with you. Your sympathetic nervous system has triggered an acute fight-or-flight alarm. In this state, your brain floods your bloodstream with stress hormones and treats racing, catastrophic thoughts as imminent emergencies—even when there is no physical danger present in your room.\n\nLet us steady your breath together to signal to your amygdala that you are safe in this second. Inhale gently through your nose for a count of four, pause softly, and exhale slowly through slightly parted lips for a count of six. As you exhale, imagine the air carrying that urgent vibration out of your chest. Gravity is holding you securely against the ground.\n\nPlace one hand on your heart and feel the steady rhythm beneath your palm. Can you name three simple objects in your immediate surroundings that are completely still?"
    },
    {
        "user": "I work 14 hours every single day. If I stop working for even an hour, an overwhelming wave of guilt washes over me telling me I am wasting my life.",
        "assistant": "You are describing what schema therapy identifies as an unrelenting standards schema—an internal psychological mechanism where human worth becomes conditionally chained to continuous output. In this framework, rest is perceived not as biological restoration, but as a dangerous liability.\n\nWhen you push your mind and body past exhaustion out of guilt, you are not exercising healthy discipline; you are depleting the very cognitive and emotional reserves required to sustain your life. Rest is not a moral reward that must be earned through suffering; it is fundamental physiological maintenance.\n\nIf you were to offer yourself the grace to suspend that internal critic for just this evening, what is one peaceful, non-productive activity your body is quietly longing for?"
    },
    {
        "user": "I got promoted to lead this team, but every day I walk into the office feeling like an absolute fraud. It is only a matter of time before everyone realizes I have no idea what I am doing.",
        "assistant": "The experience of feeling like an imposter is one of the most painful and pervasive cognitive distortions experienced by high-functioning, conscientious individuals. It arises when there is a mismatch between how you evaluate your own internal doubts—which are unfiltered and noisy—and how you perceive the external confidence of your colleagues, which is curated.\n\nWhen we step into expanded responsibility, our brains naturally encounter unfamiliar territory. Your mind interprets that lack of total certainty as evidence of incompetence, rather than what it truly is: the natural friction of growth. Nobody who takes on significant challenges has all the answers in advance.\n\nIf someone you deeply respect came to you expressing this identical doubt about their abilities, would you conclude they were a fraud, or would you recognize that they care deeply about doing meaningful work?"
    }
]

print("Fetching open clinical psychology dialogues from samhog/psychology-10k...")
clinical_dialogues = []
try:
    url = "https://huggingface.co/datasets/samhog/psychology-10k/resolve/main/Psychology-10K.json"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=20) as resp:
        data = json.loads(resp.read().decode("utf-8"))
        print(f"Loaded {len(data)} dialogues from psychology-10k")
        for item in data:
            u = item.get("input", "").strip()
            a = item.get("output", "").strip()
            if u and a and len(u) > 15 and len(a) > 35:
                clinical_dialogues.append({"user": u, "assistant": a})
except Exception as e:
    print(f"Note on remote dataset fetch: {e}")

all_pairs = CLINICAL_PROTOCOLS + clinical_dialogues
print(f"Total training dialogue pairs: {len(all_pairs):,}")

system_prompt = (
    "You are High on Therapy, a senior clinical psychologist, somatic regulation specialist, "
    "and confidential therapeutic counselor. Maintain a calm, articulate, dignified presence, "
    "grounded in CBT, ACT, Polyvagal theory, and somatic mindfulness."
)

with open(CHATML_FILE, "w", encoding="utf-8") as f_jsonl, open(CORPUS_FILE, "w", encoding="utf-8") as f_txt:
    for pair in all_pairs:
        u = pair["user"].strip()
        a = pair["assistant"].strip()
        chatml_entry = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": u},
                {"role": "assistant", "content": a}
            ]
        }
        f_jsonl.write(json.dumps(chatml_entry, ensure_ascii=False) + "\n")
        f_txt.write(f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{u}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>\n\n")

print(f"Compiled dataset: {CHATML_FILE} ({os.path.getsize(CHATML_FILE):,} bytes)")
print(f"Compiled raw corpus: {CORPUS_FILE} ({os.path.getsize(CORPUS_FILE):,} bytes)")



## 4. Train 100% Native BPE Tokenizer


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from transformers import PreTrainedTokenizerFast

TOKENIZER_DIR = os.path.join(DRIVE_CHECKPOINT_DIR, "tokenizer")
os.makedirs(TOKENIZER_DIR, exist_ok=True)

print("Training native BPE tokenizer on counseling corpus...")
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

special_tokens = [
    "<unk>", "<s>", "</s>", "<pad>",
    "<|im_start|>", "<|im_end|>", "<|thought|>", "<|somatic_note|>"
]

trainer = BpeTrainer(
    vocab_size=16384,
    min_frequency=2,
    special_tokens=special_tokens,
    initial_alphabet=ByteLevel.alphabet()
)

tokenizer.train([CORPUS_FILE], trainer)

raw_path = os.path.join(TOKENIZER_DIR, "tokenizer.json")
tokenizer.save(raw_path)

chat_template = (
    "{% for message in messages %}"
    "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)

fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=raw_path,
    bos_token="<s>",
    eos_token="<|im_end|>",
    unk_token="<unk>",
    pad_token="<pad>",
    chat_template=chat_template
)

fast_tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"Native Tokenizer successfully saved to Google Drive: {TOKENIZER_DIR}")
print(f"Vocabulary Size: {len(fast_tokenizer):,}")



## 5. Initialize Model from Scratch (Select Architectural Tier)


In [ ]:
# @title 5. Initialize Model from Scratch (Select Architectural Tier) { display-mode: "form" }
from transformers import LlamaConfig, LlamaForCausalLM

# ==========================================
# SELECT YOUR SCALE TIER:
# "nano"  -> ~135M params (Fast, 4-8 hrs on T4)
# "base"  -> ~360M params (Balanced, 12-24 hrs on T4/L4/A100)
# "prime" -> ~1.1B params (Deep clinical mastery, multi-day)
# "ultra" -> ~3.2B params (Massive scale)
# ==========================================
TIER = "base"  # @param ["nano", "base", "prime", "ultra"]

TIER_CONFIGS = {
    "nano":  {"hidden": 768,  "intermediate": 2048, "layers": 12, "heads": 12, "kv_heads": 4},
    "base":  {"hidden": 1024, "intermediate": 2816, "layers": 16, "heads": 16, "kv_heads": 4},
    "prime": {"hidden": 2048, "intermediate": 5632, "layers": 22, "heads": 32, "kv_heads": 8},
    "ultra": {"hidden": 3072, "intermediate": 8192, "layers": 28, "heads": 32, "kv_heads": 8},
}

spec = TIER_CONFIGS[TIER]
vocab_size = len(fast_tokenizer)

print(f"Configuring Native High on Therapy Model ({TIER.upper()} TIER)...")
config = LlamaConfig(
    vocab_size=vocab_size,
    hidden_size=spec["hidden"],
    intermediate_size=spec["intermediate"],
    num_hidden_layers=spec["layers"],
    num_attention_heads=spec["heads"],
    num_key_value_heads=spec["kv_heads"],
    max_position_embeddings=4096,
    hidden_act="silu",
    initializer_range=0.02,
    rms_norm_eps=1e-6,
    rope_theta=10000.0,
    tie_word_embeddings=True,
    pad_token_id=3,
    bos_token_id=1,
    eos_token_id=5,
    use_cache=True,
)

# Initialize from random weights
print(f"Initializing {TIER.upper()} tier weights from random Gaussian distribution (0 external weights)...")
model = LlamaForCausalLM(config)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,} ({total_params / 1e6:.2f} Million)")
print("100% of parameters are freshly initialized and ready for training!")


## 6. Multi-Day Pre-Training with Auto-Backup & Hardware Auto-Upgrader
- ⏰ **5-Minute Auto-Backup**: Automatically saves the model & tokenizer to Google Drive every 5 minutes (`autobackups/autobackup_latest`).
- 🚀 **Hardware Auto-Upgrader**: Checks available GPU VRAM and automatically scales batch size and worker threads to maximize GPU saturation.
- 🛡️ **Auto-Resume**: Detects the latest checkpoint on Drive and resumes immediately after any Colab reset.



In [ ]:
import os
import glob
import time
import shutil
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback

MODEL_OUTPUT_DIR = os.path.join(DRIVE_CHECKPOINT_DIR, f"model_{TIER}")
BACKUP_DIR = os.path.join(DRIVE_CHECKPOINT_DIR, f"autobackups_{TIER}")
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

# -------------------------------------------------------------
# 1. HARDWARE AUTO-UPGRADER (MAXIMUM PHYSICAL VRAM SATURATION)
# -------------------------------------------------------------
USE_GRADIENT_CHECKPOINTING = False

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    free_b, total_b = torch.cuda.mem_get_info()
    total_gb = total_b / (1024**3)
    free_gb = free_b / (1024**3)
    print(f"📊 [HARDWARE SCAN] GPU: {gpu_name} | VRAM: {total_gb:.1f} GB Total, {free_gb:.1f} GB Free")
    
    if total_gb >= 35.0:  # A100 (40GB/80GB) -> 35-70 GB consumption
        batch_size = 64
        grad_accum = 1
        workers = 4
    elif total_gb >= 20.0:  # L4 (24GB)
        batch_size = 48
        grad_accum = 1
        workers = 4
    elif total_gb >= 14.0:  # Tesla T4 (14.6GB)
        # Increased from 24 -> 32 to consume the remaining 3.4 GB free space (~13.6-14.0 GB)
        batch_size = 32
        grad_accum = 1
        workers = 4
    else:
        batch_size = 12
        grad_accum = 2
        workers = 2
        
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    eff_batch = batch_size * grad_accum
    print(f"🚀 [MAX VRAM SATURATION] batch_size={batch_size}, grad_accum={grad_accum}, grad_checkpointing={USE_GRADIENT_CHECKPOINTING}, workers={workers}")
else:
    batch_size = 4
    grad_accum = 8
    workers = 0

# -------------------------------------------------------------
# 2. 5-MINUTE PERIODIC AUTO-BACKUP CALLBACK
# -------------------------------------------------------------
class TimedAutoBackupCallback(TrainerCallback):
    def __init__(self, interval_seconds=300, target_dir=BACKUP_DIR, tokenizer=fast_tokenizer):
        self.interval = interval_seconds
        self.last_backup = time.time()
        self.target_dir = target_dir
        self.tokenizer = tokenizer

    def on_step_end(self, args, state, control, model=None, **kwargs):
        now = time.time()
        if now - self.last_backup >= self.interval:
            self.last_backup = now
            step = state.global_step
            timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
            step_dir = os.path.join(self.target_dir, f"backup_step_{step}")
            latest_dir = os.path.join(self.target_dir, "autobackup_latest")
            
            print(f"\n💾 [5-MIN AUTO-BACKUP] Triggered at step {step} ({timestamp}). Saving to Google Drive...")
            try:
                if model is not None:
                    model.save_pretrained(step_dir)
                    model.save_pretrained(latest_dir)
                if self.tokenizer is not None:
                    self.tokenizer.save_pretrained(step_dir)
                    self.tokenizer.save_pretrained(latest_dir)
                    
                backups = glob.glob(os.path.join(self.target_dir, "backup_step_*"))
                if len(backups) > 2:
                    backups.sort(key=lambda x: int(x.split("_")[-1]) if x.split("_")[-1].isdigit() else 0)
                    for old in backups[:-2]:
                        shutil.rmtree(old, ignore_errors=True)
                print(f"✅ [5-MIN AUTO-BACKUP] Successfully secured to Google Drive: {latest_dir}")
            except Exception as e:
                print(f"⚠️ [5-MIN AUTO-BACKUP] Backup note: {e}")

# -------------------------------------------------------------
# 3. RUNTIME RESOURCE MONITOR CALLBACK
# -------------------------------------------------------------
class RuntimeResourceMonitorCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 50 == 0 and torch.cuda.is_available():
            free_b, total_b = torch.cuda.mem_get_info()
            free_gb = free_b / (1024**3)
            total_gb = total_b / (1024**3)
            pct = ((total_b - free_b) / total_b) * 100
            print(f"📊 [RESOURCE MONITOR - Step {state.global_step}] VRAM: {total_gb - free_gb:.1f}GB / {total_gb:.1f}GB ({pct:.1f}% used, {free_gb:.1f}GB free)")
            if free_gb > total_gb * 0.5:
                print(f"⚡ [AUTO-UPGRADER] Hardware has {free_gb:.1f}GB surplus VRAM headroom. Compute throughput running at peak stability!")

# -------------------------------------------------------------
# 4. DATASET & DYNAMIC PADDING COLLATOR SETUP
# -------------------------------------------------------------
class HighOnTherapyDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        self.examples = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                messages = item.get("messages", [])
                if messages:
                    text = tokenizer.apply_chat_template(messages, tokenize=False)
                    encoded = tokenizer(text, truncation=True, max_length=max_length)
                    if len(encoded["input_ids"]) > 10:
                        self.examples.append(list(encoded["input_ids"]))
        print(f"Loaded {len(self.examples):,} tokenized sequences.")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return {"input_ids": self.examples[idx]}

def dynamic_pad_collator(batch):
    pad_id = fast_tokenizer.pad_token_id if fast_tokenizer.pad_token_id is not None else 3
    max_len = max(len(x["input_ids"]) for x in batch)
    padded_inputs, padded_labels, attn_mask = [], [], []
    for item in batch:
        ids = item["input_ids"]
        pad_len = max_len - len(ids)
        padded_inputs.append(ids + [pad_id] * pad_len)
        padded_labels.append(ids + [-100] * pad_len)
        attn_mask.append([1] * len(ids) + [0] * pad_len)
    return {
        "input_ids": torch.tensor(padded_inputs, dtype=torch.long),
        "labels": torch.tensor(padded_labels, dtype=torch.long),
        "attention_mask": torch.tensor(attn_mask, dtype=torch.long)
    }

train_dataset = HighOnTherapyDataset(CHATML_FILE, fast_tokenizer, max_length=1024)
data_collator = dynamic_pad_collator

# -------------------------------------------------------------
# 5. AUTO-RESUME CHECKPOINT DETECTION (WITH AUTO-CLEANUP)
# -------------------------------------------------------------
checkpoints = glob.glob(os.path.join(MODEL_OUTPUT_DIR, "checkpoint-*"))
valid_checkpoints = []
for cp in checkpoints:
    if os.path.isfile(os.path.join(cp, "trainer_state.json")):
        valid_checkpoints.append(cp)
    else:
        print(f"⚠️ [AUTO-CLEANUP] Detected incomplete checkpoint: {cp}. Removing to prevent crash...")
        shutil.rmtree(cp, ignore_errors=True)

resume_path = None
if valid_checkpoints:
    valid_checkpoints.sort(key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0)
    resume_path = valid_checkpoints[-1]
    print(f"\n>>> [AUTO-RESUME ACTIVE] Found verified checkpoint: {resume_path}")
    print("Resuming training seamlessly from exact last step!\n")
else:
    latest_backup = os.path.join(BACKUP_DIR, "autobackup_latest")
    if os.path.isdir(latest_backup) and os.path.isfile(os.path.join(latest_backup, "model.safetensors")):
        print(f"\n>>> [AUTO-RESUME ACTIVE] Loading model weights from latest 5-min backup: {latest_backup}")
        try:
            model = LlamaForCausalLM.from_pretrained(latest_backup)
            print("Successfully loaded model weights from 5-min backup! Resuming pre-training...")
        except Exception as e:
            print(f"Backup load note: {e}")
    else:
        print(f"\n>>> Starting new {TIER.upper()} pre-training run from step 0...\n")

# -------------------------------------------------------------
# 6. TRAINING CONFIGURATION & EXECUTION
# -------------------------------------------------------------
model.config.use_cache = False if USE_GRADIENT_CHECKPOINTING else True

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=3e-4 if TIER == "base" else (5e-4 if TIER == "nano" else 1.5e-4),
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    bf16=False,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    dataloader_num_workers=workers,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none"
)

import inspect
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "data_collator": data_collator,
    "callbacks": [
        TimedAutoBackupCallback(interval_seconds=300),
        RuntimeResourceMonitorCallback()
    ]
}
sig = inspect.signature(Trainer.__init__).parameters
if "processing_class" in sig:
    trainer_kwargs["processing_class"] = fast_tokenizer
elif "tokenizer" in sig:
    trainer_kwargs["tokenizer"] = fast_tokenizer

trainer = Trainer(**trainer_kwargs)

# Start / Resume training
trainer.train(resume_from_checkpoint=resume_path)

# Restore use_cache before saving
model.config.use_cache = True

# Final Save
print(f"\nSaving final bootstrapped {TIER.upper()} model to {MODEL_OUTPUT_DIR}...")
trainer.save_model(MODEL_OUTPUT_DIR)
fast_tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print(f"Pre-training for {TIER.upper()} completed and saved successfully!")


## 7. Live Clinical Generation Test


In [ ]:
from transformers import pipeline

print("Loading trained model for clinical testing...")
pipe = pipeline(
    "text-generation",
    model=MODEL_OUTPUT_DIR,
    tokenizer=MODEL_OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1
)

def test_counselor(client_concern):
    messages = [
        {"role": "system", "content": "You are High on Therapy, a senior clinical psychologist and confidential therapeutic counselor."},
        {"role": "user", "content": client_concern}
    ]
    prompt = fast_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, max_new_tokens=250, temperature=0.7, top_p=0.9, do_sample=True)
    generated = out[0]["generated_text"][len(prompt):].strip()
    generated = generated.replace("<|im_end|>", "").strip()
    return generated

test_queries = [
    "I feel overwhelmed with work and completely stuck in place.",
    "I feel like a total fraud in my new job and everyone will find out.",
    "My heart is racing and I feel sudden panic out of nowhere."
]

for q in test_queries:
    print(f"\n[Client]: {q}")
    print(f"[High on Therapy Native]:\n{test_counselor(q)}")
    print("-" * 60)



## 8. Push to Hugging Face Hub


In [ ]:
print(f"Pushing native model and tokenizer to Hugging Face Hub ({HF_REPO})...")
model.push_to_hub(HF_REPO, token=HF_TOKEN)
fast_tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"\n🎉 CONGRATULATIONS! Your 100% self-trained model is live on Hugging Face: https://huggingface.co/{HF_REPO}")
print("Your High on Therapy application will now load this native model directly!")

